In [ ]:
import sys, time, subprocess
from pathlib import Path

PROJECT_ROOT = Path("/Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast").resolve()

SCRIPT  = PROJECT_ROOT / "src/dl_wind/train_xgb_multihorizon_maxlog.py"
DATASET = PROJECT_ROOT / "data/processed/station3529_schausende_1to12/features_1to12_core_clean.parquet"
SPLITS  = PROJECT_ROOT / "data/processed/station3529_schausende_1to12/splits_12h.json"

assert SCRIPT.exists(), SCRIPT
assert DATASET.exists(), DATASET
assert SPLITS.exists(), SPLITS

cmd = [
    sys.executable, str(SCRIPT),
    "--dataset", str(DATASET),
    "--splits", str(SPLITS),
    "--spot", "Schausende",
    "--project_root", str(PROJECT_ROOT),
    "--lead_min", "1",
    "--lead_max", "12",
    "--regime_conformal",
    "--save_feature_importance",
]

print("Running:\n", " ".join(cmd))
t0 = time.time()

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")

rc = proc.wait()
print("\nReturn code:", rc)
print("Duration min:", (time.time() - t0) / 60)

if rc != 0:
    raise RuntimeError("Training failed")

Running:
 /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/.venv311/bin/python /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/src/dl_wind/train_xgb_multihorizon_maxlog.py --dataset /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/data/processed/station3529_schausende_1to12/features_1to12_core_clean.parquet --splits /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/data/processed/station3529_schausende_1to12/splits_12h.json --spot Schausende --project_root /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast --lead_min 1 --lead_max 12 --regime_conformal --save_feature_importance
MULTIHORIZON_RUN: 20260306_221954_Schausende_multihorizon_f60b37
ROOT_RUN_DIR: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37
DATASET: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/data/processed/station3529_schausen

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast").resolve()
RUNS_DIR = PROJECT_ROOT / "logs" / "runs"

run_dirs = [p for p in RUNS_DIR.iterdir() if p.is_dir()]
assert run_dirs, f"No run dirs found in {RUNS_DIR}"

last_run = sorted(run_dirs, key=lambda p: p.stat().st_mtime)[-1]
print("Last run:", last_run)

summary_path = last_run / "summary_all_leads.csv"
assert summary_path.exists(), f"Missing {summary_path}"

df = pd.read_csv(summary_path).sort_values("lead_hours")
display(df)

Last run: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37


,lead_hours,speed_rmse,speed_mae,speed_bias,speed_r2,gust_rmse,gust_mae,gust_bias,gust_r2,dir_angular_mae_deg,best_n_speed,best_n_gust,best_n_dir_sin,best_n_dir_cos,lead_duration_min,conformal_speed_coverage,conformal_gust_coverage,conformal_speed_width,conformal_gust_width
0,1,1.241744,0.897269,0.391587,0.891685,2.604264,2.076786,1.815345,0.776146,0.847831,3000,1200,3000,3000,3.800301,0.434238,0.427975,1.149377,3.033700
1,2,1.130868,0.856766,0.179705,0.909065,2.440097,1.955824,1.517995,0.800214,0.910829,600,600,1500,3000,3.693672,0.739496,0.611345,2.404426,4.122157
2,3,1.322903,0.927316,0.257479,0.883283,2.574943,1.976870,1.650476,0.782721,0.906954,1500,800,1500,3000,3.787191,0.559748,0.578616,1.544861,3.746407
3,4,1.302073,0.911328,0.431093,0.881059,2.663865,2.098523,1.808510,0.766014,0.892038,1500,1200,3000,3000,3.782530,0.579498,0.451883,1.649928,3.041190
4,5,1.232029,0.917209,0.301583,0.892259,2.513126,1.929164,1.564869,0.788370,0.845769,600,600,3000,3000,3.715751,0.755789,0.661053,2.449996,4.341393
5,6,1.336309,0.897226,0.192871,0.880943,2.501052,1.904790,1.546486,0.795017,0.910828,800,800,3000,3000,3.737303,0.731092,0.602941,2.304249,3.816267
6,7,1.298490,0.923850,0.324056,0.881817,2.622331,2.053205,1.722963,0.773309,0.895433,1200,800,3000,3000,3.726049,0.593291,0.540881,1.733214,3.739085
7,8,1.212632,0.921606,0.304522,0.895798,2.515079,1.995404,1.649005,0.788256,0.953773,800,200,3000,3000,3.722436,0.723629,0.761603,2.352780,5.619785
8,9,1.375510,0.968530,0.237451,0.874069,2.626858,1.937506,1.478392,0.774159,0.936654,800,600,3000,3000,3.731723,0.690526,0.669474,2.270186,4.413445
9,10,1.361905,0.935587,0.304696,0.870129,2.790865,2.131867,1.829988,0.743331,0.896471,1200,3000,3000,3000,3.798412,0.665966,0.214286,1.935669,1.569459


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

PROJECT_ROOT = Path("/Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast").resolve()
RUNS_DIR = PROJECT_ROOT / "logs" / "runs"

# latest run
last_run = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)[-1]
summary_path = last_run / "summary_all_leads.csv"
df = pd.read_csv(summary_path).sort_values("lead_hours")

EVAL_DIR = last_run / "eval_plots"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

x = df["lead_hours"].to_numpy()

def save_multi_line(series_dict, title, ylabel, outname):
    plt.figure()
    for name, y in series_dict.items():
        plt.plot(x, y, marker="o", label=name)
    plt.title(title)
    plt.xlabel("lead_hours")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(EVAL_DIR / outname, dpi=160)
    plt.close()

# Main metrics
save_multi_line({"speed_rmse": df["speed_rmse"], "gust_rmse": df["gust_rmse"]},
                "RMSE vs Lead (Holdout)", "RMSE", "rmse_speed_gust_vs_lead.png")

save_multi_line({"speed_mae": df["speed_mae"], "gust_mae": df["gust_mae"]},
                "MAE vs Lead (Holdout)", "MAE", "mae_speed_gust_vs_lead.png")

save_multi_line({"speed_bias": df["speed_bias"], "gust_bias": df["gust_bias"]},
                "Bias (pred-true) vs Lead (Holdout)", "Bias", "bias_speed_gust_vs_lead.png")

save_multi_line({"speed_r2": df["speed_r2"], "gust_r2": df["gust_r2"]},
                "R² vs Lead (Holdout)", "R²", "r2_speed_gust_vs_lead.png")

if "dir_angular_mae_deg" in df.columns:
    plt.figure()
    plt.plot(x, df["dir_angular_mae_deg"], marker="o")
    plt.title("Direction angular MAE (deg) vs Lead (Holdout)")
    plt.xlabel("lead_hours")
    plt.ylabel("MAE (deg)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(EVAL_DIR / "dir_mae_deg_vs_lead.png", dpi=160)
    plt.close()

# Conformal
if "conformal_speed_coverage" in df.columns:
    save_multi_line({"speed_coverage": df["conformal_speed_coverage"], "gust_coverage": df["conformal_gust_coverage"]},
                    "Conformal Coverage vs Lead (Holdout)", "coverage", "coverage_speed_gust_vs_lead.png")
    save_multi_line({"speed_pi_width": df["conformal_speed_width"], "gust_pi_width": df["conformal_gust_width"]},
                    "Conformal Interval Width vs Lead (Holdout)", "avg PI width", "pi_width_speed_gust_vs_lead.png")

# Long table for paper (one row per lead+target)
rows = []
for _, r in df.iterrows():
    lead = int(r["lead_hours"])
    for t in ["speed", "gust"]:
        rows.append({
            "lead_hours": lead,
            "target": t,
            "rmse": float(r[f"{t}_rmse"]),
            "mae": float(r[f"{t}_mae"]),
            "bias": float(r[f"{t}_bias"]),
            "r2": float(r[f"{t}_r2"]),
            "coverage": float(r[f"conformal_{t}_coverage"]) if f"conformal_{t}_coverage" in df.columns else np.nan,
            "avg_pi_width": float(r[f"conformal_{t}_width"]) if f"conformal_{t}_width" in df.columns else np.nan,
        })
long_df = pd.DataFrame(rows)
long_path = last_run / "paper_table_long.csv"
long_df.to_csv(long_path, index=False)

# Small summary JSON for slides
summary_json = {
    "run_dir": str(last_run),
    "n_leads": int(df.shape[0]),
    "speed_rmse_mean": float(np.nanmean(df["speed_rmse"])),
    "gust_rmse_mean": float(np.nanmean(df["gust_rmse"])),
    "speed_mae_mean": float(np.nanmean(df["speed_mae"])),
    "gust_mae_mean": float(np.nanmean(df["gust_mae"])),
}
json_path = last_run / "summary_for_slides.json"
json_path.write_text(json.dumps(summary_json, indent=2), encoding="utf-8")

print("Wrote PNGs to:", EVAL_DIR)
print("Wrote:", long_path)
print("Wrote:", json_path)
print("PNGs:")
for p in sorted(EVAL_DIR.glob("*.png")):
    print(" -", p.name)

Wrote PNGs to: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/eval_plots
Wrote: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/paper_table_long.csv
Wrote: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/summary_for_slides.json
PNGs:
 - bias_speed_gust_vs_lead.png
 - coverage_speed_gust_vs_lead.png
 - dir_mae_deg_vs_lead.png
 - mae_speed_gust_vs_lead.png
 - pi_width_speed_gust_vs_lead.png
 - r2_speed_gust_vs_lead.png
 - rmse_speed_gust_vs_lead.png


In [4]:
from pathlib import Path
import glob

PROJECT_ROOT = Path("/Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast").resolve()
RUNS_DIR = PROJECT_ROOT / "logs" / "runs"
last_run = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)[-1]

print("Run dir:", last_run)
print("Summary:", last_run / "summary_all_leads.csv")
print("Eval plots:", last_run / "eval_plots")
print("Per-lead reports (examples):")
for p in sorted(last_run.glob("lead_*/lead_report.json"))[:3]:
    print(" -", p)

Run dir: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37
Summary: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/summary_all_leads.csv
Eval plots: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/eval_plots
Per-lead reports (examples):
 - /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/lead_01/lead_report.json
 - /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/lead_02/lead_report.json
 - /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37/lead_03/lead_report.json


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT_ROOT = Path("/Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast").resolve()
RUNS_DIR = PROJECT_ROOT / "logs" / "runs"

# Letzten Run wählen
last_run = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)[-1]
print("Last run:", last_run)

# Run-Meta laden
run_meta = json.loads((last_run / "run_meta.json").read_text())
dataset_path = PROJECT_ROOT / run_meta["dataset"]
spot = run_meta["spot"]
lead_min = run_meta["lead_min"]
lead_max = run_meta["lead_max"]
holdout_start = pd.to_datetime(run_meta["holdout_start"])

print("Dataset:", dataset_path)
print("Spot:", spot)
print("Holdout start:", holdout_start)

# Original-Datensatz laden
df = pd.read_parquet(dataset_path)
df["feature_time"] = pd.to_datetime(df["feature_time"])

if "station" in df.columns:
    df = df[df["station"].astype(str).str.contains(spot, case=False, na=False)].copy()

df = df[(df["lead_hours"] >= lead_min) & (df["lead_hours"] <= lead_max)].copy()

# Holdout filtern
df_ho = df[df["feature_time"] >= holdout_start].copy()

def rmse(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.sqrt(np.mean((y - p) ** 2)))

def mae(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.mean(np.abs(y - p)))

def bias(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.mean(p - y))

def r2(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    ss_res = np.sum((y - p) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

def angular_error_deg(y_true_deg, y_pred_deg):
    y_true_deg = np.asarray(y_true_deg, dtype=float)
    y_pred_deg = np.asarray(y_pred_deg, dtype=float)
    d = (y_pred_deg - y_true_deg + 180) % 360 - 180
    return np.abs(d)

# Spalten erkennen
speed_cols = sorted([c for c in df_ho.columns if c.startswith("fc_windspd__")])
gust_cols = sorted([c for c in df_ho.columns if c.startswith("fc_gust__")])
dir_cols = sorted([c for c in df_ho.columns if c.startswith("fc_winddir__")])

print("Speed model cols:", speed_cols)
print("Gust model cols:", gust_cols)
print("Dir model cols:", dir_cols)

rows = []

for lead in range(lead_min, lead_max + 1):
    dlead = df_ho[df_ho["lead_hours"] == lead].copy()

    # ML predictions laden
    pred_path = last_run / "reports" / "predictions" / last_run.name / f"lead_{lead:02d}" / "summary_all_leads.csv"
    if not pred_path.exists():
        pred_path = last_run / f"lead_{lead:02d}" / "predictions_holdout_all.csv"

    ml_speed_rmse = np.nan
    ml_gust_rmse = np.nan
    ml_dir_mae = np.nan

    if pred_path.exists():
        pred = pd.read_csv(pred_path)
        pred["feature_time"] = pd.to_datetime(pred["feature_time"])
        speed_true_col = "speed_true" if "speed_true" in pred.columns else "y_speed_true"
        speed_pred_col = "speed_pred" if "speed_pred" in pred.columns else "y_speed_pred"
        gust_true_col = "gust_true" if "gust_true" in pred.columns else "y_gust_true"
        gust_pred_col = "gust_pred" if "gust_pred" in pred.columns else "y_gust_pred"
        dir_true_col = "dir_true_deg" if "dir_true_deg" in pred.columns else "y_dir_true_deg"
        dir_pred_col = "dir_pred_deg" if "dir_pred_deg" in pred.columns else "y_dir_pred_deg"

        if speed_true_col in pred.columns and speed_pred_col in pred.columns:
            ml_speed_rmse = rmse(pred[speed_true_col], pred[speed_pred_col])
        if gust_true_col in pred.columns and gust_pred_col in pred.columns:
            ml_gust_rmse = rmse(pred[gust_true_col], pred[gust_pred_col])
        if dir_true_col in pred.columns and dir_pred_col in pred.columns:
            ml_dir_mae = float(np.mean(angular_error_deg(pred[dir_true_col], pred[dir_pred_col])))

    # Einzelne NWP-Modelle
    for c in speed_cols:
        tmp = dlead[["obs_speed_mean", c]].dropna()
        if len(tmp) > 0:
            rows.append({
                "lead_hours": lead,
                "target": "speed",
                "model": c.replace("fc_windspd__", ""),
                "rmse": rmse(tmp["obs_speed_mean"], tmp[c]),
                "mae": mae(tmp["obs_speed_mean"], tmp[c]),
                "bias": bias(tmp["obs_speed_mean"], tmp[c]),
                "r2": r2(tmp["obs_speed_mean"], tmp[c]),
                "rows": len(tmp),
            })

    for c in gust_cols:
        tmp = dlead[["obs_gust_max", c]].dropna()
        if len(tmp) > 0:
            rows.append({
                "lead_hours": lead,
                "target": "gust",
                "model": c.replace("fc_gust__", ""),
                "rmse": rmse(tmp["obs_gust_max"], tmp[c]),
                "mae": mae(tmp["obs_gust_max"], tmp[c]),
                "bias": bias(tmp["obs_gust_max"], tmp[c]),
                "r2": r2(tmp["obs_gust_max"], tmp[c]),
                "rows": len(tmp),
            })

    for c in dir_cols:
        tmp = dlead[["obs_dir_deg", c]].dropna()
        if len(tmp) > 0:
            rows.append({
                "lead_hours": lead,
                "target": "direction",
                "model": c.replace("fc_winddir__", ""),
                "angular_mae_deg": float(np.mean(angular_error_deg(tmp["obs_dir_deg"], tmp[c]))),
                "rows": len(tmp),
            })

    # einfache Ensemble-Baselines
    if speed_cols:
        tmp = dlead[["obs_speed_mean"] + speed_cols].copy()
        tmp["mean_ensemble"] = tmp[speed_cols].mean(axis=1, skipna=True)
        tmp["median_ensemble"] = tmp[speed_cols].median(axis=1, skipna=True)

        for ens_name in ["mean_ensemble", "median_ensemble"]:
            x = tmp[["obs_speed_mean", ens_name]].dropna()
            if len(x) > 0:
                rows.append({
                    "lead_hours": lead,
                    "target": "speed",
                    "model": ens_name,
                    "rmse": rmse(x["obs_speed_mean"], x[ens_name]),
                    "mae": mae(x["obs_speed_mean"], x[ens_name]),
                    "bias": bias(x["obs_speed_mean"], x[ens_name]),
                    "r2": r2(x["obs_speed_mean"], x[ens_name]),
                    "rows": len(x),
                })

    if gust_cols:
        tmp = dlead[["obs_gust_max"] + gust_cols].copy()
        tmp["mean_ensemble"] = tmp[gust_cols].mean(axis=1, skipna=True)
        tmp["median_ensemble"] = tmp[gust_cols].median(axis=1, skipna=True)

        for ens_name in ["mean_ensemble", "median_ensemble"]:
            x = tmp[["obs_gust_max", ens_name]].dropna()
            if len(x) > 0:
                rows.append({
                    "lead_hours": lead,
                    "target": "gust",
                    "model": ens_name,
                    "rmse": rmse(x["obs_gust_max"], x[ens_name]),
                    "mae": mae(x["obs_gust_max"], x[ens_name]),
                    "bias": bias(x["obs_gust_max"], x[ens_name]),
                    "r2": r2(x["obs_gust_max"], x[ens_name]),
                    "rows": len(x),
                })

    if dir_cols:
        tmp = dlead[["obs_dir_deg"] + dir_cols].copy()
        tmp["mean_dir"] = tmp[dir_cols].mean(axis=1, skipna=True)
        x = tmp[["obs_dir_deg", "mean_dir"]].dropna()
        if len(x) > 0:
            rows.append({
                "lead_hours": lead,
                "target": "direction",
                "model": "mean_ensemble",
                "angular_mae_deg": float(np.mean(angular_error_deg(x["obs_dir_deg"], x["mean_dir"]))),
                "rows": len(x),
            })

    # ML als eigene Zeile
    rows.append({
        "lead_hours": lead,
        "target": "speed",
        "model": "xgboost",
        "rmse": ml_speed_rmse,
    })
    rows.append({
        "lead_hours": lead,
        "target": "gust",
        "model": "xgboost",
        "rmse": ml_gust_rmse,
    })
    rows.append({
        "lead_hours": lead,
        "target": "direction",
        "model": "xgboost",
        "angular_mae_deg": ml_dir_mae,
    })

cmp_df = pd.DataFrame(rows)
cmp_df.head(20)

Last run: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/logs/runs/20260306_221954_Schausende_multihorizon_f60b37
Dataset: /Users/filipbertram/Hochschule/Master/Deep Learning/dl_wind_forecast/data/processed/station3529_schausende_1to12/features_1to12_core_clean.parquet
Spot: Schausende
Holdout start: 2025-12-26 12:00:00
Speed model cols: ['fc_windspd__gfs', 'fc_windspd__harmeu', 'fc_windspd__icon', 'fc_windspd__iconeu', 'fc_windspd__ifs', 'fc_windspd__metno', 'fc_windspd__wrfeuh']
Gust model cols: ['fc_gust__gfs', 'fc_gust__harmeu', 'fc_gust__icon', 'fc_gust__iconeu', 'fc_gust__ifs', 'fc_gust__metno', 'fc_gust__wrfeuh']
Dir model cols: []


,lead_hours,target,model,rmse,mae,bias,r2,rows,angular_mae_deg
0,1,speed,gfs,6.919031,6.106527,5.850209,-2.247401,239.0,NaN
1,1,speed,harmeu,5.297977,4.734937,4.108954,-0.967795,478.0,NaN
2,1,speed,iconeu,5.007664,4.427757,3.931866,-0.754375,477.0,NaN
3,1,speed,ifs,4.645231,4.203193,3.571849,-0.460021,238.0,NaN
4,1,speed,metno,6.935886,6.006820,5.601464,-2.265288,239.0,NaN
5,1,speed,wrfeuh,4.787248,4.178826,3.603174,-0.816694,230.0,NaN
6,1,gust,gfs,11.215910,10.083389,9.959121,-3.129148,239.0,NaN
7,1,gust,harmeu,10.608602,9.343996,9.223117,-2.709703,478.0,NaN
8,1,gust,iconeu,8.493172,7.451698,7.306289,-1.372806,477.0,NaN
9,1,gust,ifs,8.564852,7.598866,7.452143,-1.401311,238.0,NaN
